# EditGuard: full 50-sample evaluation with three displayed examples

EditGuard was run on **all 50 clips in the fixed MOSE selection**. To keep the notebook readable, it displays the same three fixed examples used by the other notebooks; the aggregate table below and linked evidence cover all 50.

```text
original -> framewise EditGuard embedding -> watermarked video -> ProPainter, DiffuEraser, or EffectErase -> EditGuard localization
```

The released EditGuard app resizes every input image to 512 x 512. The video adapter therefore applies the released image method independently to each frame after the same 512 x 512 resize, and aligns the object masks to that size.

EditGuard's released residual threshold is 0.2. White predicted pixels are locations where at least one recovered residual channel exceeds that threshold. The notebook also reports mean recovery accuracy for the fixed 64-bit copyright message.


## Metric guide

| Term | Meaning | Interpretation |
| --- | --- | --- |
| Embedding PSNR | Similarity between the aligned source frames and watermarked frames. | Higher is better; it measures embedding distortion, not detector quality. |
| Global pixel IoU | Total intersection divided by total union across all frames and pixels. | 1 is perfect overlap. 0 means no overlap. This is the primary localization result. |
| Mean frame IoU | IoU calculated for each frame and then averaged. | Empty ground-truth and prediction masks score 1, so global IoU remains the primary score. |
| Effective GT mask | The removal mask after the remover's actual dilation and spatial preprocessing. | This is the correct reference mask for IoU. |
| Predicted mask | The method's own detector output after documented binarization. | White indicates predicted tampering. |
| Message bit accuracy | Fraction of fixed copyright bits recovered correctly, averaged across frames. | Available for EditGuard; not exposed by OmniGuard's public demo. |
| Runtime | Wall-clock time recorded on the shared experiment hardware. | Useful for this run, not a hardware-independent benchmark. |
| SHA-256 | File fingerprint stored in the JSON record. | Confirms exact file identity. It is not a quality metric. |


## Aggregate results across all 50 selected clips

All 50 clips completed embedding, ProPainter, DiffuEraser, EffectErase, and native localization. The values below are unweighted means of the 50 per-clip results; the complete arrays are in [watermark-summary.json](../../evidence/mose-task1/review50/watermark-summary.json).

> **Four-inpainter scope:** this is a historical 50-sample aggregate for the first three removers. E2FGVI-HQ was run for the required one-clip matrix, so an aggregate value is not comparable and is not fabricated.

| Metric | ProPainter | DiffuEraser | EffectErase | E2FGVI-HQ |
| --- | ---: | ---: | ---: | ---: |
| Mean global pixel IoU | **0.0871** | **0.0890** | **0.0709** | not comparable |
| Mean 64-bit recovery accuracy | 0.7235 | 0.6824 | 0.5002 | not comparable |

Mean embedding PSNR across all 50 clips: **36.7968 dB**.

The remainder of this notebook deliberately displays only three clips: `02b72b8d`, `0ab75563`, and `0b02886a`.


## Notebook setup

After a fresh clone, run `python3 scripts/prepare_mose_notebooks.py` from the repository root. This downloads and verifies the videos shared by the method notebooks. Run the next cell once to enable the video, image, download, and metadata viewers. If Git LFS has left a pointer file in place of a video, the viewer reports the preparation command instead of displaying a broken player.


In [ ]:
import base64
from pathlib import Path
import html
import json
from IPython.display import HTML, Markdown, display

LFS_POINTER_PREFIX = b"version https://git-lfs.github.com/spec/v1"

def find_repo_root():
    for start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (start / "evidence/mose-task1/review50/manifest.json").is_file():
            return start
    raise FileNotFoundError("Run this notebook from inside the paper-reproductions checkout.")

REPO_ROOT = find_repo_root()
REVIEW_ROOT = REPO_ROOT / "evidence/mose-task1/review50"

def artifact_data_url(source, mime_type):
    if not source.is_file():
        raise FileNotFoundError(source)
    payload = source.read_bytes()
    if payload.startswith(LFS_POINTER_PREFIX):
        raise RuntimeError(
            f"{source} is a Git LFS pointer, not the real media file. "
            "From the repository root run: python3 scripts/prepare_mose_notebooks.py"
        )
    return f"data:{mime_type};base64,{base64.b64encode(payload).decode('ascii')}"

def show_videos(items, columns=3):
    cards = []
    for title, relative, explanation in items:
        if relative is None:
            cards.append(
                f"<section class='media-card'><h4>{html.escape(title)}</h4>"
                f"<p><strong>Not applicable.</strong></p><p>{html.escape(explanation)}</p></section>"
            )
            continue
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "video/mp4")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<video controls muted loop playsinline preload='metadata' src='{data_url}'></video>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this video</a></p>"
            "</section>"
        )
    display(HTML(
        f"<style>.media-grid{{display:grid;grid-template-columns:repeat({columns},minmax(220px,1fr));gap:16px}}"
        ".media-card{border:1px solid #d0d7de;border-radius:8px;padding:12px;background:#fff;color:#24292f}"
        ".media-card video{width:100%;background:#000}.media-card h4{margin:0 0 8px}.media-card p{font-size:13px}</style>"
        f"<div class='media-grid'>{''.join(cards)}</div>"
    ))

def show_images(items, columns=2):
    cards = []
    for title, relative, explanation in items:
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "image/png")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<img style='width:100%' src='{data_url}' alt='{html.escape(title)}'>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this image</a></p>"
            "</section>"
        )
    display(HTML(
        f"<div class='media-grid' style='grid-template-columns:repeat({columns},minmax(260px,1fr))'>"
        f"{''.join(cards)}</div>"
    ))

def show_metadata(paths):
    for relative in paths:
        path = REVIEW_ROOT / relative
        payload = json.loads(path.read_text())
        display(Markdown(f"#### `{relative}`"))
        print(json.dumps(payload, indent=2))

print(f"Loaded evidence from {REVIEW_ROOT}")


## Canonical one-clip comparison: all four inpainters

This is the canonical comparison for the minimum one-clip requirement. All four removers use the same frozen watermarked input and aligned binary mask. Every remover is presented in the same order, with the same output-level fields and the same video/JSON evidence. Method-specific resizing, padding, and mask dilation remain recorded in JSON rather than being silently normalized.

| Inpainting method | Status | Runtime (s) | Frames | Dimensions | FPS | Bytes | Evidence |
|---|---:|---:|---:|---:|---:|---:|---|
| ProPainter | complete | 49.509 | 67 | 512 × 512 | 25.000 | 840,896 | [video](../../evidence/mose-task1/review50/media/02b72b8d/methods/editguard/propainter.mp4) · [JSON](../../evidence/mose-task1/review50/media/02b72b8d/methods/editguard/propainter.json) |
| DiffuEraser | complete | 131.377 | 67 | 512 × 512 | 25.000 | 888,917 | [video](../../evidence/mose-task1/review50/media/02b72b8d/methods/editguard/diffueraser.mp4) · [JSON](../../evidence/mose-task1/review50/media/02b72b8d/methods/editguard/diffueraser.json) |
| EffectErase | complete | 234.490 | 67 | 512 × 512 | 25.000 | 415,508 | [video](../../evidence/mose-task1/review50/media/02b72b8d/methods/editguard/effecterase.mp4) · [JSON](../../evidence/mose-task1/review50/media/02b72b8d/methods/editguard/effecterase.json) |
| E2FGVI-HQ | complete | 21.046 | 67 | 512 × 512 | 25.000 | 883,880 | [video](../../evidence/mose-task1/review50/media/02b72b8d/methods/editguard/e2fgvi.mp4) · [JSON](../../evidence/mose-task1/review50/media/02b72b8d/methods/editguard/e2fgvi.json) |

Watermark method: **Editguard**. Runtime is shown only when the original run record captured it; `not recorded` is not replaced with an estimate. Watermark-specific localization or recovery metrics are downstream evaluations and are kept separate from this equal inpainting comparison.


In [ ]:
from pathlib import Path
import json
from IPython.display import JSON, Markdown, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/finalize_four_inpainter_notebooks.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
ATTACK = ROOT / 'evidence/mose-task1/review50/media/02b72b8d/methods/editguard'
for name in ("input-watermarked.mp4", "watermarked.mp4", "gt-mask.mp4"):
    path = ATTACK / name
    if path.is_file():
        print(name)
        display(Video(str(path), embed=True, html_attributes="controls muted loop"))

REMOVERS = (
    ("ProPainter", "propainter"),
    ("DiffuEraser", "diffueraser"),
    ("EffectErase", "effecterase"),
    ("E2FGVI-HQ", "e2fgvi"),
)
for label, stem in REMOVERS:
    display(Markdown(f"### {label}"))
    video = ATTACK / f"{stem}.mp4"
    metadata = ATTACK / f"{stem}.json"
    if not video.is_file() or not metadata.is_file():
        raise FileNotFoundError(f"Missing {label} evidence in {ATTACK}")
    display(Video(str(video), embed=True, html_attributes="controls muted loop"))
    display(JSON(json.loads(metadata.read_text())))

for name in ("sample.json", "removal-pair.json"):
    metadata = ATTACK / name
    if metadata.is_file():
        print(name)
        display(JSON(json.loads(metadata.read_text())))


## 1. Sample `02b72b8d`


### Source and unwatermarked removal reference

The source has 67 frames at 960 x 480, 25 FPS, and a duration of about 2.68 seconds. The raw mask marks the object selected for removal. The three unwatermarked outputs show how ProPainter, DiffuEraser, and EffectErase behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/02b72b8d/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/02b72b8d/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/02b72b8d/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/02b72b8d/diffueraser.mp4) | [EffectErase reference](../../evidence/mose-task1/review50/media/02b72b8d/baseline/effecterase.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/02b72b8d/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/02b72b8d/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/02b72b8d/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/02b72b8d/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.'), ('EffectErase without watermark', 'media/02b72b8d/baseline/effecterase.mp4', 'Unwatermarked EffectErase removal reference.')])


### EditGuard results

Every displayed method video has 512 x 512 frames. Embedding took 21.296 s. The embedding PSNR is **35.9017 dB**, with MSE 16.7076 and MAE 3.5244.

> **Four-inpainter scope:** E2FGVI-HQ is shown in this table. Its common inpainting metrics come from the recorded one-clip run; downstream watermark localization/recovery was not rerun and is marked `not evaluated`.

| Metric | ProPainter | DiffuEraser | EffectErase | E2FGVI-HQ |
| --- | ---: | ---: | ---: | ---: |
| Removal runtime | 49.509 s | 131.377 s | 234.490 s | 21.046 s |
| Global pixel IoU | **0.0276** | **0.0297** | **0.0168** | not evaluated |
| Mean frame IoU | 0.0276 | 0.0297 | 0.0168 | not evaluated |
| Minimum frame IoU | 0.0226 | 0.0245 | 0.0143 | not evaluated |
| Maximum frame IoU | 0.0334 | 0.0351 | 0.0198 | not evaluated |
| Intersection pixels | 484,318 | 520,937 | 295,818 | not evaluated |
| Union pixels | 17,563,648 | 17,563,648 | 17,563,648 | not evaluated |
| Mean 64-bit recovery accuracy | 0.8242 | 0.7164 | 0.5149 | not evaluated |

The localization record contains all three attacked videos. Its latest detector pass took 14.402 s. The global IoU compares the displayed binary prediction with the effective mask actually used by each remover. For this sample, the released EditGuard threshold marks the full displayed frame as tampered after all three removal pipelines. The low IoU therefore equals the fraction of the full frame occupied by the effective removal mask; it is not a missing-mask result.


In [ ]:
show_videos([('EditGuard input', 'media/02b72b8d/methods/editguard/aligned-original.mp4', 'Source resized to the released EditGuard 512 x 512 input protocol.'), ('EditGuard aligned mask', 'media/02b72b8d/methods/editguard/gt-mask.mp4', "Selected-object mask aligned to EditGuard's 512 x 512 frames."), ('EditGuard watermarked', 'media/02b72b8d/methods/editguard/watermarked.mp4', 'Watermarked video before object removal.'), ('ProPainter attacked video', 'media/02b72b8d/methods/editguard/propainter.mp4', 'ProPainter output from the EditGuard-watermarked input.'), ('ProPainter effective GT', 'media/02b72b8d/methods/editguard/effective-gt-mask-propainter.mp4', 'Mask after ProPainter preprocessing; the IoU reference.'), ('ProPainter predicted mask', 'media/02b72b8d/methods/editguard/pred-mask-propainter.mp4', 'Native EditGuard localization after ProPainter.'), ('DiffuEraser attacked video', 'media/02b72b8d/methods/editguard/diffueraser.mp4', 'DiffuEraser output from the EditGuard-watermarked input.'), ('DiffuEraser effective GT', 'media/02b72b8d/methods/editguard/effective-gt-mask-diffueraser.mp4', 'Mask after DiffuEraser preprocessing; the IoU reference.'), ('DiffuEraser predicted mask', 'media/02b72b8d/methods/editguard/pred-mask-diffueraser.mp4', 'Native EditGuard localization after DiffuEraser.'), ('EffectErase attacked video', 'media/02b72b8d/methods/editguard/effecterase.mp4', 'EffectErase output from the EditGuard-watermarked input.'), ('EffectErase effective GT', 'media/02b72b8d/methods/editguard/effective-gt-mask-effecterase.mp4', 'Mask after EffectErase bilinear preprocessing; the IoU reference.'), ('EffectErase predicted mask', 'media/02b72b8d/methods/editguard/pred-mask-effecterase.mp4', 'Native EditGuard localization after EffectErase.')])


### Run records

The next cell prints the embedding, removal, and localization JSON records, including checkpoints, repository commits, thresholds, runtimes, per-frame values, and file fingerprints.


In [ ]:
show_metadata(['media/02b72b8d/methods/editguard/embed.json', 'media/02b72b8d/methods/editguard/propainter.json', 'media/02b72b8d/methods/editguard/diffueraser.json', 'media/02b72b8d/methods/editguard/effecterase.json', 'media/02b72b8d/methods/editguard/localization.json'])


## 2. Sample `0ab75563`


### Source and unwatermarked removal reference

The source has 104 frames at 960 x 448, 25 FPS, and a duration of about 4.16 seconds. The raw mask marks the object selected for removal. The three unwatermarked outputs show how ProPainter, DiffuEraser, and EffectErase behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/0ab75563/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/0ab75563/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/0ab75563/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/0ab75563/diffueraser.mp4) | [EffectErase reference](../../evidence/mose-task1/review50/media/0ab75563/baseline/effecterase.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/0ab75563/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/0ab75563/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/0ab75563/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/0ab75563/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.'), ('EffectErase without watermark', 'media/0ab75563/baseline/effecterase.mp4', 'Unwatermarked EffectErase removal reference.')])


### EditGuard results

Every displayed method video has 512 x 512 frames. Embedding took 33.489 s. The embedding PSNR is **38.2324 dB**, with MSE 9.7689 and MAE 2.3917.

> **Four-inpainter scope:** only the first fixed sample was required and run for E2FGVI-HQ; this additional sample is marked `not run`.

| Metric | ProPainter | DiffuEraser | EffectErase | E2FGVI-HQ |
| --- | ---: | ---: | ---: | ---: |
| Removal runtime | 70.006 s | 169.037 s | 275.527 s | not run |
| Global pixel IoU | **0.0121** | **0.0121** | **0.0090** | not run |
| Mean frame IoU | 0.0121 | 0.0121 | 0.0090 | not run |
| Minimum frame IoU | 0.0000 | 0.0000 | 0.0000 | not run |
| Maximum frame IoU | 0.2384 | 0.2401 | 0.2158 | not run |
| Intersection pixels | 329,133 | 331,240 | 246,655 | not run |
| Union pixels | 27,262,976 | 27,262,976 | 27,262,976 | not run |
| Mean 64-bit recovery accuracy | 0.8428 | 0.7809 | 0.5705 | not run |

The localization record contains all three attacked videos. Its latest detector pass took 21.466 s. The global IoU compares the displayed binary prediction with the effective mask actually used by each remover. For this sample, the released EditGuard threshold marks the full displayed frame as tampered after all three removal pipelines. The low IoU therefore equals the fraction of the full frame occupied by the effective removal mask; it is not a missing-mask result.


In [ ]:
show_videos([('EditGuard input', 'media/0ab75563/methods/editguard/aligned-original.mp4', 'Source resized to the released EditGuard 512 x 512 input protocol.'), ('EditGuard aligned mask', 'media/0ab75563/methods/editguard/gt-mask.mp4', "Selected-object mask aligned to EditGuard's 512 x 512 frames."), ('EditGuard watermarked', 'media/0ab75563/methods/editguard/watermarked.mp4', 'Watermarked video before object removal.'), ('ProPainter attacked video', 'media/0ab75563/methods/editguard/propainter.mp4', 'ProPainter output from the EditGuard-watermarked input.'), ('ProPainter effective GT', 'media/0ab75563/methods/editguard/effective-gt-mask-propainter.mp4', 'Mask after ProPainter preprocessing; the IoU reference.'), ('ProPainter predicted mask', 'media/0ab75563/methods/editguard/pred-mask-propainter.mp4', 'Native EditGuard localization after ProPainter.'), ('DiffuEraser attacked video', 'media/0ab75563/methods/editguard/diffueraser.mp4', 'DiffuEraser output from the EditGuard-watermarked input.'), ('DiffuEraser effective GT', 'media/0ab75563/methods/editguard/effective-gt-mask-diffueraser.mp4', 'Mask after DiffuEraser preprocessing; the IoU reference.'), ('DiffuEraser predicted mask', 'media/0ab75563/methods/editguard/pred-mask-diffueraser.mp4', 'Native EditGuard localization after DiffuEraser.'), ('EffectErase attacked video', 'media/0ab75563/methods/editguard/effecterase.mp4', 'EffectErase output from the EditGuard-watermarked input.'), ('EffectErase effective GT', 'media/0ab75563/methods/editguard/effective-gt-mask-effecterase.mp4', 'Mask after EffectErase bilinear preprocessing; the IoU reference.'), ('EffectErase predicted mask', 'media/0ab75563/methods/editguard/pred-mask-effecterase.mp4', 'Native EditGuard localization after EffectErase.')])


### Run records

The next cell prints the embedding, removal, and localization JSON records, including checkpoints, repository commits, thresholds, runtimes, per-frame values, and file fingerprints.


In [ ]:
show_metadata(['media/0ab75563/methods/editguard/embed.json', 'media/0ab75563/methods/editguard/propainter.json', 'media/0ab75563/methods/editguard/diffueraser.json', 'media/0ab75563/methods/editguard/effecterase.json', 'media/0ab75563/methods/editguard/localization.json'])


## 3. Sample `0b02886a`


### Source and unwatermarked removal reference

The source has 30 frames at 960 x 640, 25 FPS, and a duration of about 1.20 seconds. The raw mask marks the object selected for removal. The three unwatermarked outputs show how ProPainter, DiffuEraser, and EffectErase behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/0b02886a/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/0b02886a/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/0b02886a/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/0b02886a/diffueraser.mp4) | [EffectErase reference](../../evidence/mose-task1/review50/media/0b02886a/baseline/effecterase.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/0b02886a/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/0b02886a/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/0b02886a/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/0b02886a/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.'), ('EffectErase without watermark', 'media/0b02886a/baseline/effecterase.mp4', 'Unwatermarked EffectErase removal reference.')])


### EditGuard results

Every displayed method video has 512 x 512 frames. Embedding took 9.339 s. The embedding PSNR is **36.2304 dB**, with MSE 15.4895 and MAE 3.2409.

> **Four-inpainter scope:** only the first fixed sample was required and run for E2FGVI-HQ; this additional sample is marked `not run`.

| Metric | ProPainter | DiffuEraser | EffectErase | E2FGVI-HQ |
| --- | ---: | ---: | ---: | ---: |
| Removal runtime | 26.380 s | 64.375 s | 104.039 s | not run |
| Global pixel IoU | **0.1023** | **0.1057** | **0.0786** | not run |
| Mean frame IoU | 0.1023 | 0.1057 | 0.0786 | not run |
| Minimum frame IoU | 0.0662 | 0.0690 | 0.0461 | not run |
| Maximum frame IoU | 0.1556 | 0.1591 | 0.1298 | not run |
| Intersection pixels | 804,730 | 831,438 | 617,979 | not run |
| Union pixels | 7,864,320 | 7,864,320 | 7,864,320 | not run |
| Mean 64-bit recovery accuracy | 0.8177 | 0.7297 | 0.5240 | not run |

The localization record contains all three attacked videos. Its latest detector pass took 6.135 s. The global IoU compares the displayed binary prediction with the effective mask actually used by each remover. For this sample, the released EditGuard threshold marks the full displayed frame as tampered after all three removal pipelines. The low IoU therefore equals the fraction of the full frame occupied by the effective removal mask; it is not a missing-mask result.


In [ ]:
show_videos([('EditGuard input', 'media/0b02886a/methods/editguard/aligned-original.mp4', 'Source resized to the released EditGuard 512 x 512 input protocol.'), ('EditGuard aligned mask', 'media/0b02886a/methods/editguard/gt-mask.mp4', "Selected-object mask aligned to EditGuard's 512 x 512 frames."), ('EditGuard watermarked', 'media/0b02886a/methods/editguard/watermarked.mp4', 'Watermarked video before object removal.'), ('ProPainter attacked video', 'media/0b02886a/methods/editguard/propainter.mp4', 'ProPainter output from the EditGuard-watermarked input.'), ('ProPainter effective GT', 'media/0b02886a/methods/editguard/effective-gt-mask-propainter.mp4', 'Mask after ProPainter preprocessing; the IoU reference.'), ('ProPainter predicted mask', 'media/0b02886a/methods/editguard/pred-mask-propainter.mp4', 'Native EditGuard localization after ProPainter.'), ('DiffuEraser attacked video', 'media/0b02886a/methods/editguard/diffueraser.mp4', 'DiffuEraser output from the EditGuard-watermarked input.'), ('DiffuEraser effective GT', 'media/0b02886a/methods/editguard/effective-gt-mask-diffueraser.mp4', 'Mask after DiffuEraser preprocessing; the IoU reference.'), ('DiffuEraser predicted mask', 'media/0b02886a/methods/editguard/pred-mask-diffueraser.mp4', 'Native EditGuard localization after DiffuEraser.'), ('EffectErase attacked video', 'media/0b02886a/methods/editguard/effecterase.mp4', 'EffectErase output from the EditGuard-watermarked input.'), ('EffectErase effective GT', 'media/0b02886a/methods/editguard/effective-gt-mask-effecterase.mp4', 'Mask after EffectErase bilinear preprocessing; the IoU reference.'), ('EffectErase predicted mask', 'media/0b02886a/methods/editguard/pred-mask-effecterase.mp4', 'Native EditGuard localization after EffectErase.')])


### Run records

The next cell prints the embedding, removal, and localization JSON records, including checkpoints, repository commits, thresholds, runtimes, per-frame values, and file fingerprints.


In [ ]:
show_metadata(['media/0b02886a/methods/editguard/embed.json', 'media/0b02886a/methods/editguard/propainter.json', 'media/0b02886a/methods/editguard/diffueraser.json', 'media/0b02886a/methods/editguard/effecterase.json', 'media/0b02886a/methods/editguard/localization.json'])


## Validation and provenance

- The full-review validator decoded all 50 EditGuard result sets and checked frame counts, dimensions, FPS, mask foreground, and displayed-mask IoU against the JSON result.
- Full 50-sample validation report: [validation.json](../../evidence/mose-task1/review50/validation.json)
- Three-example notebook validation: [image-method-notebook-validation.json](../../evidence/mose-task1/review50/image-method-notebook-validation.json)
- Full 50-sample aggregate metrics: [watermark-summary.json](../../evidence/mose-task1/review50/watermark-summary.json)
- Experimental contract: [PROTOCOL.md](../../evidence/mose-task1/PROTOCOL.md)
- Official repository: [EditGuard](https://github.com/xuanyuzhang21/EditGuard)
- DiffuEraser implementation used for every removal result: [lixiaowen-xw/DiffuEraser](https://github.com/lixiaowen-xw/DiffuEraser)
- EffectErase implementation used for every removal result: [FudanCVL/EffectErase](https://github.com/FudanCVL/EffectErase)

The displayed values come from the committed EditGuard evidence records.
